In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

DATA_PATH = Path("credit_card_transactions.csv")
df_raw = pd.read_csv(DATA_PATH)

In [2]:
df_raw.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,merch_zipcode
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0,28705.0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0,NaN
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0,83236.0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0,NaN
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0,22844.0


# Feature Engineering de Riesgo Transaccional

Se construye un dataset enriquecido con variables de monto (A), temporalidad (T) y ubicación (L), manteniendo también las variables originales.

In [3]:
# Calculo de features y construccion del dataset final enriquecido (look-back estricto)
EPS = 1e-9


df = df_raw.copy()

# Estandarizacion minima
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")
df["amt"] = pd.to_numeric(df["amt"], errors="coerce")
df = df.dropna(subset=["cc_num", "trans_date_trans_time", "amt"]).copy()

# Ubicacion: pais real si existe; de lo contrario proxy regional (state)
if "country" in df.columns:
    df["location_country"] = df["country"].astype(str)
elif "state" in df.columns:
    df["location_country"] = df["state"].astype(str)
else:
    df["location_country"] = "UNK"

df["location_city"] = df["city"].astype(str) if "city" in df.columns else "UNK"
df["location_key"] = df["location_country"] + "|" + df["location_city"]

# Orden temporal por cliente
sort_cols = ["cc_num", "trans_date_trans_time"]
if "trans_num" in df.columns:
    sort_cols.append("trans_num")
df = df.sort_values(sort_cols).reset_index(drop=True)

g_user = df.groupby("cc_num", sort=False)
user_txn_idx = g_user.cumcount() + 1
user_txn_prev = (user_txn_idx - 1).clip(lower=0)

# ============================
# 1) Variables de monto (A1-A16)
# Look-back estricto: solo historial previo
# ============================
prev_amt = g_user["amt"].shift(1)
df["A1_amt_diff_prev"] = (df["amt"] - prev_amt).abs().fillna(0.0)

# Historial acumulado previo por usuario
user_cum_sum = g_user["amt"].cumsum()
user_cum_sum_prev = user_cum_sum - df["amt"]
user_mean_prev = np.where(user_txn_prev > 0, user_cum_sum_prev / user_txn_prev, np.nan)

user_median_cum = (
    g_user["amt"]
    .expanding()
    .median()
    .reset_index(level=0, drop=True)
)
user_median_prev = user_median_cum.groupby(df["cc_num"], sort=False).shift(1)

# Fallback robusto para primeras transacciones
global_amt_median = df["amt"].median()
user_mean_prev_s = pd.Series(user_mean_prev, index=df.index).fillna(global_amt_median)
user_median_prev_s = user_median_prev.fillna(global_amt_median)

df["A2_amt_user_cum_mean"] = user_mean_prev_s
df["A3_amt_dev_user_cum_median"] = (df["amt"] - user_median_prev_s).abs()

# A4: montos identicos en 24h por usuario (solo pasado)
A4_tmp = (
    df.set_index("trans_date_trans_time")
      .groupby(["cc_num", "amt"])["amt"]
      .rolling("24h")
      .count()
      .reset_index(level=[0, 1], drop=True)
)
df["A4_identical_amt_cnt_24h"] = (A4_tmp.values - 1).clip(min=0)

# Estadisticas previas por usuario (media/std)
user_mean_cum = g_user["amt"].expanding().mean().reset_index(level=0, drop=True)
user_std_cum = g_user["amt"].expanding().std().reset_index(level=0, drop=True).fillna(0.0)

user_mean_prev2 = user_mean_cum.groupby(df["cc_num"], sort=False).shift(1).fillna(global_amt_median)
user_std_prev = user_std_cum.groupby(df["cc_num"], sort=False).shift(1).fillna(0.0)

df["A5_amt_user_std_score"] = (df["amt"] - user_mean_prev2).abs() / (user_std_prev + EPS)
df["A6_amt_within_1std_user"] = ((df["amt"] >= (user_mean_prev2 - user_std_prev)) & (df["amt"] <= (user_mean_prev2 + user_std_prev))).astype(int)

# Estadisticas previas por comercio (no globales con futuro)
if "merchant" not in df.columns:
    df["merchant"] = "UNK"

g_merch = df.groupby("merchant", sort=False)
merch_txn_idx = g_merch.cumcount() + 1
merch_prev_cnt = (merch_txn_idx - 1).clip(lower=0)
merch_cum_sum = g_merch["amt"].cumsum()
merch_cum_sum_prev = merch_cum_sum - df["amt"]
merch_mean_prev = np.where(merch_prev_cnt > 0, merch_cum_sum_prev / merch_prev_cnt, np.nan)
merch_mean_prev_s = pd.Series(merch_mean_prev, index=df.index).fillna(global_amt_median)

merch_std_cum = g_merch["amt"].expanding().std().reset_index(level=0, drop=True).fillna(0.0)
merch_std_prev = merch_std_cum.groupby(df["merchant"], sort=False).shift(1).fillna(0.0)

df["A7_amt_diff_merchant_mean"] = (df["amt"] - merch_mean_prev_s).abs()
df["A8_merchant_amt_std"] = merch_std_prev

# Usuario-comercio look-back
g_um = df.groupby(["cc_num", "merchant"], sort=False)
um_txn_idx = g_um.cumcount() + 1
um_prev_cnt = (um_txn_idx - 1).clip(lower=0)
um_cum_sum = g_um["amt"].cumsum()
um_cum_sum_prev = um_cum_sum - df["amt"]
um_cum_mean_prev = np.where(um_prev_cnt > 0, um_cum_sum_prev / um_prev_cnt, np.nan)
um_cum_mean_prev_s = pd.Series(um_cum_mean_prev, index=df.index).fillna(global_amt_median)

um_cum_std = g_um["amt"].expanding().std().reset_index(level=[0, 1], drop=True).fillna(0.0)
um_cum_std_prev = um_cum_std.groupby([df["cc_num"], df["merchant"]], sort=False).shift(1).fillna(0.0)

df["A9_user_merchant_cum_mean"] = um_cum_mean_prev_s
df["A10_user_merchant_cum_std"] = um_cum_std_prev
df["A11_user_merchant_cum_sum"] = um_cum_sum_prev.clip(lower=0)

# Mantener nombres A12-A14 por compatibilidad, pero calculados solo con pasado
df["A12_user_merchant_global_mean"] = df["A9_user_merchant_cum_mean"]
df["A13_user_merchant_global_std"] = df["A10_user_merchant_cum_std"]
df["A14_user_merchant_global_sum"] = df["A11_user_merchant_cum_sum"]

df["A15_amt_within_2std_user_merchant"] = ((df["amt"] >= (df["A9_user_merchant_cum_mean"] - 2 * df["A10_user_merchant_cum_std"])) & (df["amt"] <= (df["A9_user_merchant_cum_mean"] + 2 * df["A10_user_merchant_cum_std"]))).astype(int)

# Refinamiento: confianza mas estricta y con historial minimo previo
df["A16_amt_within_confidence_user_merchant"] = (((df["amt"] >= (df["A9_user_merchant_cum_mean"] - df["A10_user_merchant_cum_std"])) & (df["amt"] <= (df["A9_user_merchant_cum_mean"] + df["A10_user_merchant_cum_std"]))) & (um_prev_cnt >= 3)).astype(int)

# ===================================
# 2) Variables temporales (T1-T9)
# ===================================
prev_time = g_user["trans_date_trans_time"].shift(1)
df["T1_time_diff_sec"] = (df["trans_date_trans_time"] - prev_time).dt.total_seconds().fillna(np.nan)

t1_safe = df["T1_time_diff_sec"].fillna(np.inf).clip(lower=1.0)
df["T2_amt_time_angle"] = np.arctan2(df["amt"], t1_safe)

hour = df["trans_date_trans_time"].dt.hour
hour_angle = 2 * np.pi * (hour / 24.0)
df["T3_hour_sin"] = np.sin(hour_angle)
df["T4_hour_cos"] = np.cos(hour_angle)

df["T5_is_madrugada"] = (hour < 6).astype(int)

# Ratio acumulado previo de madrugada
user_madrugada_cum = g_user["T5_is_madrugada"].cumsum()
user_madrugada_prev = user_madrugada_cum - df["T5_is_madrugada"]
df["T6_user_madrugada_ratio_cum"] = np.where(user_txn_prev > 0, user_madrugada_prev / user_txn_prev, 0.0)

# Proxy Von Mises acumulado previo (R)
user_cum_sin = g_user["T3_hour_sin"].cumsum()
user_cum_cos = g_user["T4_hour_cos"].cumsum()
user_cum_sin_prev = user_cum_sin - df["T3_hour_sin"]
user_cum_cos_prev = user_cum_cos - df["T4_hour_cos"]
df["T7_user_hour_vonmises_proxy"] = np.where(
    user_txn_prev > 0,
    np.sqrt(user_cum_sin_prev.pow(2) + user_cum_cos_prev.pow(2)) / user_txn_prev,
    0.0,
)

# T8: conteos en ventanas moviles por usuario (solo pasado)
for window, colname in [
    ("1s", "T8_txn_count_1s"),
    ("10s", "T8_txn_count_10s"),
    ("60s", "T8_txn_count_60s"),
    ("1h", "T8_txn_count_1h"),
    ("2h", "T8_txn_count_2h"),
]:
    tmp = (
        df.set_index("trans_date_trans_time")
          .groupby("cc_num")["amt"]
          .rolling(window)
          .count()
          .reset_index(level=0, drop=True)
    )
    df[colname] = (tmp.values - 1).clip(min=0)

# T9: coherencia del intervalo temporal vs historial previo
user_t1_mean = g_user["T1_time_diff_sec"].expanding().mean().reset_index(level=0, drop=True)
user_t1_std = g_user["T1_time_diff_sec"].expanding().std().reset_index(level=0, drop=True).fillna(0.0)

user_t1_mean_prev = user_t1_mean.groupby(df["cc_num"], sort=False).shift(1)
user_t1_std_prev = user_t1_std.groupby(df["cc_num"], sort=False).shift(1).fillna(0.0)

df["T9_time_interval_coherent"] = ((df["T1_time_diff_sec"] >= (user_t1_mean_prev - 2 * user_t1_std_prev)) & (df["T1_time_diff_sec"] <= (user_t1_mean_prev + 2 * user_t1_std_prev))).fillna(0).astype(int)

# ==============================
# 3) Variables espaciales (L1-L6)
# ==============================
# Conteos previos (solo pasado)
df["L1_user_city_visit_count"] = df.groupby(["cc_num", "location_city"], sort=False).cumcount()
df["L2_user_country_visit_count"] = df.groupby(["cc_num", "location_country"], sort=False).cumcount()

new_city_flag = (~df.duplicated(subset=["cc_num", "location_city"])).astype(int)
new_country_flag = (~df.duplicated(subset=["cc_num", "location_country"])).astype(int)

# Unicos previos: acumulado de "primeras veces" hasta el registro anterior
city_unique_cum = new_city_flag.groupby(df["cc_num"]).cumsum()
country_unique_cum = new_country_flag.groupby(df["cc_num"]).cumsum()

df["L3_user_unique_city_count"] = (city_unique_cum - new_city_flag).clip(lower=0)
df["L3_user_unique_country_count"] = (country_unique_cum - new_country_flag).clip(lower=0)

df["L4_user_city_txn_ratio"] = np.where(user_txn_prev > 0, df["L1_user_city_visit_count"] / user_txn_prev, 0.0)
df["L4_user_country_txn_ratio"] = np.where(user_txn_prev > 0, df["L2_user_country_visit_count"] / user_txn_prev, 0.0)

prev_loc = df.groupby("cc_num", sort=False)["location_key"].shift(1)
df["L5_same_location_as_prev"] = (df["location_key"] == prev_loc).fillna(False).astype(int)
df["L6_location_seen_before"] = df.duplicated(subset=["cc_num", "location_key"]).astype(int)

# Dataset final: variables originales + nuevas features
feature_cols = [c for c in df.columns if c.startswith(("A", "T", "L"))]
df_features = df.copy()

print(f"Filas del dataset final: {df_features.shape[0]:,}")
print(f"Columnas totales del dataset final: {df_features.shape[1]:,}")
print(f"Nuevas features agregadas: {len(feature_cols)}")

print("\nVista de nuevas features:")
display(df_features[feature_cols].head())

output_path = Path("credit_card_transactions_fe.csv")
df_features.to_csv(output_path, index=False)
print(f"\nDataset enriquecido guardado en: {output_path.resolve()}")

Filas del dataset final: 1,296,675
Columnas totales del dataset final: 64
Nuevas features agregadas: 37

Vista de nuevas features:


,A1_amt_diff_prev,A2_amt_user_cum_mean,A3_amt_dev_user_cum_median,A4_identical_amt_cnt_24h,A5_amt_user_std_score,A6_amt_within_1std_user,A7_amt_diff_merchant_mean,A8_merchant_amt_std,A9_user_merchant_cum_mean,A10_user_merchant_cum_std,...,T8_txn_count_2h,T9_time_interval_coherent,L1_user_city_visit_count,L2_user_country_visit_count,L3_user_unique_city_count,L3_user_unique_country_count,L4_user_city_txn_ratio,L4_user_country_txn_ratio,L5_same_location_as_prev,L6_location_seen_before
0,0.00,47.520,40.250,0.0,4.025000e+10,0,40.25,0.0,47.52,0.0,...,0.0,0,0,0,0,0,0.0,0.0,0,0
1,45.67,7.270,45.670,0.0,4.567000e+10,0,5.42,0.0,47.52,0.0,...,0.0,0,1,1,1,1,1.0,1.0,1,1
2,29.14,30.105,51.975,0.0,1.609454e+00,0,34.56,0.0,47.52,0.0,...,1.0,0,2,2,1,1,1.0,1.0,1,1
3,47.29,47.430,18.150,0.0,3.352061e-01,1,12.73,0.0,47.52,0.0,...,0.0,1,3,3,1,1,1.0,1.0,1,1
4,7.61,44.270,16.685,0.0,5.437388e-01,1,20.34,0.0,47.52,0.0,...,1.0,1,4,4,1,1,1.0,1.0,1,1



Dataset enriquecido guardado en: C:\Users\nicog\Documents\Proyectos Personales\FraudShield\credit_card_transactions_fe.csv


In [4]:
# Diccionario de features engineered (nombres interpretables)
feature_dictionary = {
    # A: Monto
    "A1_amt_diff_prev": "Diferencia absoluta de monto vs transaccion anterior del usuario",
    "A2_amt_user_cum_mean": "Promedio acumulado previo de monto por usuario (look-back)",
    "A3_amt_dev_user_cum_median": "Desviacion del monto actual vs mediana previa del usuario",
    "A4_identical_amt_cnt_24h": "Conteo previo (24h) de montos identicos por usuario",
    "A5_amt_user_std_score": "Distancia estandarizada del monto vs perfil previo del usuario",
    "A6_amt_within_1std_user": "Bandera: monto dentro de +/-1 desviacion estandar previa del usuario",
    "A7_amt_diff_merchant_mean": "Diferencia absoluta del monto vs promedio previo del comercio",
    "A8_merchant_amt_std": "Desviacion estandar previa de montos en el comercio",
    "A9_user_merchant_cum_mean": "Promedio previo de monto usuario-comercio",
    "A10_user_merchant_cum_std": "Desviacion estandar previa de monto usuario-comercio",
    "A11_user_merchant_cum_sum": "Suma previa de monto usuario-comercio",
    "A12_user_merchant_global_mean": "Promedio previo historico usuario-comercio (look-back)",
    "A13_user_merchant_global_std": "Desviacion estandar previa historica usuario-comercio",
    "A14_user_merchant_global_sum": "Suma previa historica usuario-comercio",
    "A15_amt_within_2std_user_merchant": "Bandera: monto dentro de +/-2 desviaciones previas en usuario-comercio",
    "A16_amt_within_confidence_user_merchant": "Bandera refinada de consistencia de monto en usuario-comercio con historial minimo previo",

    # T: Temporal
    "T1_time_diff_sec": "Segundos desde la transaccion anterior del usuario",
    "T2_amt_time_angle": "atan2(monto, tiempo) para detectar montos altos en poco tiempo",
    "T3_hour_sin": "Transformacion seno de la hora (ciclica 24h)",
    "T4_hour_cos": "Transformacion coseno de la hora (ciclica 24h)",
    "T5_is_madrugada": "Bandera: transaccion entre 00:00 y 05:59",
    "T6_user_madrugada_ratio_cum": "Proporcion acumulada previa de transacciones de madrugada por usuario",
    "T7_user_hour_vonmises_proxy": "Concentracion circular acumulada previa de horarios (proxy Von Mises)",
    "T8_txn_count_1s": "Conteo previo de transacciones por usuario en ventana movil de 1 segundo",
    "T8_txn_count_10s": "Conteo previo de transacciones por usuario en ventana movil de 10 segundos",
    "T8_txn_count_60s": "Conteo previo de transacciones por usuario en ventana movil de 60 segundos",
    "T8_txn_count_1h": "Conteo previo de transacciones por usuario en ventana movil de 1 hora",
    "T8_txn_count_2h": "Conteo previo de transacciones por usuario en ventana movil de 2 horas",
    "T9_time_interval_coherent": "Bandera: intervalo temporal coherente vs historial previo del usuario",

    # L: Ubicacion
    "L1_user_city_visit_count": "Conteo previo de visitas del usuario a la ciudad actual",
    "L2_user_country_visit_count": "Conteo previo de visitas del usuario al pais/region actual",
    "L3_user_unique_city_count": "Conteo previo de ciudades unicas del usuario",
    "L3_user_unique_country_count": "Conteo previo de paises/regiones unicos del usuario",
    "L4_user_city_txn_ratio": "Proporcion previa de transacciones del usuario en la ciudad actual",
    "L4_user_country_txn_ratio": "Proporcion previa de transacciones del usuario en el pais/region actual",
    "L5_same_location_as_prev": "Bandera: misma ubicacion que la transaccion inmediatamente anterior",
    "L6_location_seen_before": "Bandera: la ubicacion actual ya aparecio antes en el historial del usuario"
}

feature_dictionary_df = (
    pd.DataFrame(
        [{"feature": k, "descripcion": v} for k, v in feature_dictionary.items()]
    )
)

print(f"Total de nuevas features definidas: {feature_dictionary_df.shape[0]}")
display(feature_dictionary_df)

Total de nuevas features definidas: 37


,feature,descripcion
0,A1_amt_diff_prev,Diferencia absoluta de monto vs transaccion an...
1,A2_amt_user_cum_mean,Promedio acumulado previo de monto por usuario...
2,A3_amt_dev_user_cum_median,Desviacion del monto actual vs mediana previa ...
3,A4_identical_amt_cnt_24h,Conteo previo (24h) de montos identicos por us...
4,A5_amt_user_std_score,Distancia estandarizada del monto vs perfil pr...
5,A6_amt_within_1std_user,Bandera: monto dentro de +/-1 desviacion estan...
6,A7_amt_diff_merchant_mean,Diferencia absoluta del monto vs promedio prev...
7,A8_merchant_amt_std,Desviacion estandar previa de montos en el com...
8,A9_user_merchant_cum_mean,Promedio previo de monto usuario-comercio
9,A10_user_merchant_cum_std,Desviacion estandar previa de monto usuario-co...
